# Grafos Computacionales y Autograd en PyTorch

## ¿Qué es un Grafo Computacional?

Un grafo computacional es una representación de cálculos matemáticos como un grafo dirigido, donde:
- Los **nodos** representan operaciones matemáticas o variables
- Las **aristas** representan el flujo de datos entre operaciones

PyTorch construye estos grafos dinámicamente durante la ejecución del código, lo que permite una gran flexibilidad en el diseño de modelos de aprendizaje profundo.

Vamos a explorar cómo PyTorch utiliza grafos computacionales con un ejemplo sencillo.

In [1]:
import torch
import matplotlib.pyplot as plt
import numpy as np

# Configuramos la visualización
%matplotlib inline

## Ejemplo: Función Matemática Sencilla

Exploraremos la función $f(x, y, z) = x * (y - z)$ y cómo PyTorch construye su grafo computacional.

In [2]:
# Creamos las variables con requires_grad=True para que PyTorch registre operaciones
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(5.0, requires_grad=True)
z = torch.tensor(3.0, requires_grad=True)

# Definimos operaciones intermedias para visualizar el grafo
w = y - z           # w = 5 - 3 = 2
f = x * w           # f = 2 * 2 = 4

print(f"x = {x.item()}")
print(f"y = {y.item()}")
print(f"z = {z.item()}")
print(f"w = y - z = {w.item()}")
print(f"f = x * w = {f.item()}")

x = 2.0
y = 5.0
z = 3.0
w = y - z = 2.0
f = x * w = 4.0


### Visualización del Grafo Computacional

Aquí está la representación del grafo computacional para nuestra función:

```
   x     y     z
   |     |     |
   |     \   /
   |      sub (w = y - z)
   |     /
   \   /
    mul (f = x * w)
     |
     f
```

Este grafo muestra cómo PyTorch registra las operaciones realizadas sobre los tensores.

## Cálculo de Gradientes con Autograd

La magia de PyTorch está en su capacidad para calcular automáticamente derivadas parciales mediante retropropagación (backpropagation). Veamos cómo Autograd calcula los gradientes:

In [3]:
# Calculamos los gradientes llamando a .backward() en la salida
f.backward()

# Revisamos los gradientes calculados
print(f"df/dx = {x.grad.item()}")  # df/dx = (y - z) = 2
print(f"df/dy = {y.grad.item()}")  # df/dy = x = 2
print(f"df/dz = {z.grad.item()}")  # df/dz = -x = -2

df/dx = 2.0
df/dy = 2.0
df/dz = -2.0


### Explicación del Cálculo de Gradientes

Para la función $f(x, y, z) = x * (y - z)$, podemos calcular las derivadas parciales analíticamente:

1. $\frac{\partial f}{\partial x} = (y - z) = 2$
2. $\frac{\partial f}{\partial y} = x = 2$
3. $\frac{\partial f}{\partial z} = -x = -2$

Los valores calculados por PyTorch coinciden con estos cálculos analíticos.

## Regla de la Cadena en Grafos Computacionales

La retropropagación aplica la regla de la cadena del cálculo. Veamos cómo funciona con un ejemplo más complejo:

In [4]:
# Reiniciamos el grafo
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(5.0, requires_grad=True)
z = torch.tensor(3.0, requires_grad=True)

# Función más compleja: f(x, y, z) = (x * y - z)^2
a = x * y       # a = 2 * 5 = 10
b = a - z       # b = 10 - 3 = 7
f = b ** 2      # f = 7^2 = 49

print(f"Valores intermedios:")
print(f"a = x * y = {a.item()}")
print(f"b = a - z = {b.item()}")
print(f"f = b^2 = {f.item()}")

# Calculamos gradientes
f.backward()

print("\nGradientes:")
print(f"df/dx = {x.grad.item()}")  
print(f"df/dy = {y.grad.item()}")  
print(f"df/dz = {z.grad.item()}")  

Valores intermedios:
a = x * y = 10.0
b = a - z = 7.0
f = b^2 = 49.0

Gradientes:
df/dx = 70.0
df/dy = 28.0
df/dz = -14.0


### Explicación de los Gradientes Mediante la Regla de la Cadena

Para la función $f(x, y, z) = (x * y - z)^2$, podemos calcular las derivadas parciales usando la regla de la cadena:

1. $\frac{\partial f}{\partial b} = 2b = 2 * 7 = 14$

2. $\frac{\partial b}{\partial a} = 1$
3. $\frac{\partial b}{\partial z} = -1$

4. $\frac{\partial a}{\partial x} = y = 5$
5. $\frac{\partial a}{\partial y} = x = 2$

Aplicando la regla de la cadena:

- $\frac{\partial f}{\partial x} = \frac{\partial f}{\partial b} \cdot \frac{\partial b}{\partial a} \cdot \frac{\partial a}{\partial x} = 14 \cdot 1 \cdot 5 = 70$
- $\frac{\partial f}{\partial y} = \frac{\partial f}{\partial b} \cdot \frac{\partial b}{\partial a} \cdot \frac{\partial a}{\partial y} = 14 \cdot 1 \cdot 2 = 28$
- $\frac{\partial f}{\partial z} = \frac{\partial f}{\partial b} \cdot \frac{\partial b}{\partial z} = 14 \cdot (-1) = -14$

## Accediendo a Gradientes Intermedios

Podemos acceder a los gradientes de las variables intermedias si las creamos con `requires_grad=True`:

In [5]:
# Recreamos nuestro grafo, pero ahora retenemos gradientes para todos los cálculos
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(5.0, requires_grad=True)
z = torch.tensor(3.0, requires_grad=True)

# Creamos operaciones intermedias y guardamos las variables
w = y - z       # w = 5 - 3 = 2
w.retain_grad() # Guarda el gradiente de w
f = x * w       # f = 2 * 2 = 4
f.retain_grad() # Guarda el gradiente de f

# Calculamos gradientes
f.backward()

print("Gradientes calculados:")
print(f"df/df = {f.grad.item()}")  # Gradiente de f respecto a sí mismo es 1
print(f"df/dw = {w.grad.item()}")  # df/dw = x = 2
print(f"df/dx = {x.grad.item()}")  # df/dx = w = 2
print(f"df/dy = {y.grad.item()}")  # df/dy = x = 2
print(f"df/dz = {z.grad.item()}")  # df/dz = -x = -2

Gradientes calculados:
df/df = 1.0
df/dw = 2.0
df/dx = 2.0
df/dy = 2.0
df/dz = -2.0


## La Importancia de los Grafos Computacionales en Deep Learning

Los grafos computacionales son fundamentales en el aprendizaje profundo porque:

1. **Cálculo Eficiente**: Permiten calcular gradientes de manera eficiente a través de la retropropagación.
2. **Aprendizaje por Gradiente**: Facilitan la optimización de modelos mediante descenso por gradiente.
3. **Reutilización de Cálculos**: Evitan recalcular valores intermedios durante el entrenamiento.
4. **Paralelización**: Facilitan la paralelización de cálculos en GPU.
5. **Flexibilidad**: Permiten crear y modificar modelos dinámicamente durante la ejecución.

PyTorch utiliza un paradigma de "define-by-run" donde el grafo se construye dinámicamente, lo que facilita la depuración y experimentación con modelos complejos.

## Ejemplo con Redes Neuronales

Veamos cómo los grafos computacionales se aplican en un modelo de red neuronal simple:

In [6]:
# Definimos una pequeña red neuronal con una capa oculta
# Entrada: x, Salida: y
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

# Pesos y bias iniciales
w1 = torch.tensor([0.1, 0.2, 0.3], requires_grad=True)
w2 = torch.tensor([0.4], requires_grad=True)
b1 = torch.tensor([0.5], requires_grad=True)
b2 = torch.tensor([0.6], requires_grad=True)

# Forward pass
# Primera capa
z1 = torch.sum(x * w1) + b1    # z1 = 0.1*1 + 0.2*2 + 0.3*3 + 0.5 = 1.6
a1 = torch.relu(z1)            # a1 = max(0, 1.6) = 1.6

# Segunda capa (salida)
z2 = a1 * w2 + b2              # z2 = 1.6*0.4 + 0.6 = 1.24
y_pred = z2                    # y_pred = 1.24

# Valor objetivo
y_true = torch.tensor([1.0])

# Función de pérdida (error cuadrático)
loss = (y_pred - y_true) ** 2  # loss = (1.24 - 1.0)^2 = 0.0576

# Cálculo de gradientes
loss.backward()

print("Valores de la red neuronal:")
print(f"z1 = {z1.item():.4f}")
print(f"a1 = {a1.item():.4f}")
print(f"z2 = {z2.item():.4f}")
print(f"y_pred = {y_pred.item():.4f}")
print(f"loss = {loss.item():.4f}")

print("\nGradientes de los parámetros:")
print(f"dL/dw1 = {w1.grad}")
print(f"dL/db1 = {b1.grad.item():.4f}")
print(f"dL/dw2 = {w2.grad.item():.4f}")
print(f"dL/db2 = {b2.grad.item():.4f}")

Valores de la red neuronal:
z1 = 1.9000
a1 = 1.9000
z2 = 1.3600
y_pred = 1.3600
loss = 0.1296

Gradientes de los parámetros:
dL/dw1 = tensor([0.2880, 0.5760, 0.8640])
dL/db1 = 0.2880
dL/dw2 = 1.3680
dL/db2 = 0.7200


## Conclusiones

1. **Grafos Computacionales**: Son representaciones de cálculos matemáticos como grafos dirigidos.
2. **Autograd**: PyTorch registra automáticamente las operaciones para calcular gradientes mediante retropropagación.
3. **Regla de la Cadena**: La retropropagación aplica la regla de la cadena para calcular gradientes eficientemente.
4. **Aplicaciones**: Los grafos computacionales son fundamentales para el entrenamiento de modelos de aprendizaje profundo.

Al entender cómo funcionan los grafos computacionales y Autograd, podemos aprovechar al máximo las capacidades de PyTorch para el aprendizaje profundo.